# Ranking predictions

**Goal**: the day before a season starts, predict the final ranking of every team in its conference
(two rankings of ~15 teams per year, ordered by points).

Pipeline:
1. Load `data_prepared/teams.csv` and build the features (only past information) - `ranking_utils.py`
2. Define the naive baselines and the models
3. **Walk-forward evaluation**: for each test season, train on all earlier seasons and predict it
4. Compare models (rank error, Spearman, playoff-set overlap), test significance against the naive model
5. Declare the best model and show its predicted rankings

Features come from `ranking_analysis.ipynb` (`ranking_selected_features.json`).

> The previous LSTM matchup approach of this notebook was kept in `ranking_predictions_lstm.ipynb`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, VotingRegressor

import ranking_utils as ru

RS = ru.RANDOM_STATE
USE_SELECTED_FEATURES = True     # features chosen in ranking_analysis.ipynb

## 1. Data loading and preprocessing

In [ ]:
teams = ru.load_teams()
df = ru.build_dataset(teams)
groups = ru.feature_groups(df)

selection_file = Path("ranking_selected_features.json")
if USE_SELECTED_FEATURES and selection_file.exists():
    FEATURES = json.loads(selection_file.read_text())["features"]
    print("Features from ranking_analysis.ipynb:", FEATURES)
else:
    FEATURES = ru.flatten(groups)
    print("Using all candidate features:", FEATURES)

EVAL_YEARS = sorted(int(y) for y in df.loc[df["year"] >= ru.TEST_START_YEAR, "year"].unique())
print(f"\n{len(df)} team-seasons | evaluation seasons: {EVAL_YEARS}")

# --- sanity checks -----------------------------------------------------------
assert not df[FEATURES].isna().any().any(), "NaN left in features"
assert df.duplicated(["year", "tmID"]).sum() == 0
leak = [c for c in FEATURES if c in ("target", "Pts_z", "conf_rank", "conf_rank_pct", "gd_z", "W_z", "GF_z", "GA_z", "made_playoffs")]
assert not leak, f"Features of the current season (leak!): {leak}"

display(df.loc[df["year"] == EVAL_YEARS[-1], ["confID", "tmID", "conf_rank", "target"] + FEATURES[:6]].head(6))

## 2. Naive baselines and models

* **Naive: previous season** - rank teams by last season's points (the carried-forward ranking, re-ranked in the current conference).
* **Naive: mean of 3 seasons** - same with the 3-season mean.
* **Ridge** - regularised linear regression (strong shrinkage / mean-reversion learner).
* **Random forest** and **Gradient boosting** - non-linear, shallow/regularised on purpose (a few hundred training rows).
* **Ensemble** - average of the three models.

All models regress the season points z-score; the ranking comes from sorting the predictions inside each conference.
Hyper-parameters are fixed and conservative: with ~30 teams per season, tuning on the test seasons would mostly fit noise.

In [ ]:
base_models = {
    "Ridge": lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30))),
    "Random forest": lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=10, max_features=0.5,
                                                    random_state=RS, n_jobs=-1),
    "Gradient boosting": lambda: HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=250,
                                                               min_samples_leaf=20, l2_regularization=1.0,
                                                               random_state=RS),
}
models = dict(base_models)
models["Ensemble (mean)"] = lambda: VotingRegressor([(name, factory()) for name, factory in base_models.items()])

baselines = {"Naive: previous season": "pts_z_lag1", "Naive: mean 3 seasons": "pts_z_mean3"}

## 3. Walk-forward evaluation

For each test season Y: fit on every season `< Y`, predict Y. Nothing from Y (or later) is used for training or
feature construction.

In [ ]:
ranked = {}
for name, column in baselines.items():
    ranked[name] = ru.rank_predictions(ru.baseline_scores(df, column, EVAL_YEARS), df)
for name, factory in models.items():
    pred = ru.walk_forward(df, factory, FEATURES, EVAL_YEARS)
    ranked[name] = ru.rank_predictions(pred, df)

season_tables = {name: ru.season_metrics(r) for name, r in ranked.items()}
summary = pd.DataFrame({name: ru.summarise(t) for name, t in season_tables.items()}).T
summary["mae_std_over_seasons"] = pd.Series({name: t["mae"].std() for name, t in season_tables.items()})
summary = summary.sort_values("mae")

n_typical = int(round(ranked["Ridge"]["conf_size"].mean()))
print(f"Evaluated {len(ranked['Ridge'])} team-seasons ({ranked['Ridge'].groupby(['year','confID']).ngroups} conference rankings).")
print(f"Reference: random ranking of {n_typical} teams -> MAE {ru.random_ranking_mae(n_typical):.2f}, "
      f"top-{ru.PLAYOFF_SPOTS} overlap {ru.PLAYOFF_SPOTS / n_typical:.2f}")
display(summary.round(3))

Metrics (averaged over the conference rankings):
* `mae` / `rmse`: error in ranking positions (lower is better)
* `spearman`: rank correlation between predicted and actual ranking (higher is better)
* `top_overlap`: share of the 8 predicted playoff teams that really finished in the top 8
* `within_2`: share of teams predicted within 2 positions of their true rank

In [ ]:
order = list(summary.index)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric, better in zip(axes, ["mae", "spearman", "top_overlap"], ["lower", "higher", "higher"]):
    values = summary.loc[order, metric]
    colors = ["#DD8452" if n.startswith("Naive") else "#4C72B0" for n in order]
    ax.barh(order[::-1], values.values[::-1], color=colors[::-1])
    for y, v in enumerate(values.values[::-1]):
        ax.text(v, y, f" {v:.3f}", va="center", fontsize=8)
    ax.set_title(f"{metric} ({better} is better)"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
per_year = {n: t.groupby("year")["mae"].mean() for n, t in season_tables.items()}
for n, s in per_year.items():
    ax.plot(s.index, s.values, marker="o", label=n, ls="--" if n.startswith("Naive") else "-")
ax.set_xlabel("Season"); ax.set_ylabel("MAE (rank positions)"); ax.set_title("Error per test season"); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4. Is the gain over the naive model real?

With only ~30 teams per season, differences can come from luck. We resample whole (season, conference) rankings
(2000 bootstrap draws) to get a 95% confidence interval for the **MAE gain over the naive "previous season" model**.
A positive interval that excludes 0 means the model is reliably better.

In [ ]:
reference = "Naive: previous season"
key = ["year", "tmID"]
base = ranked[reference].set_index(key).sort_index()
groups_id = (base["confID"] + "-" + base.index.get_level_values("year").astype(str)).to_numpy()

rows = []
for name, r in ranked.items():
    if name == reference:
        continue
    err = r.set_index(key).sort_index()["abs_err"].reindex(base.index)
    gain, (lo, hi) = ru.paired_bootstrap(base["abs_err"].to_numpy(), err.to_numpy(), groups_id)
    rows.append({"model": name, "mae_gain_vs_naive": gain, "ci95_low": lo, "ci95_high": hi,
                 "significant": lo > 0})
significance = pd.DataFrame(rows).set_index("model").sort_values("mae_gain_vs_naive", ascending=False)
display(significance.round(3))

## 5. Verdict

In [ ]:
models_only = summary.drop(index=[n for n in summary.index if n.startswith("Naive")])
best = models_only.sort_values(["mae", "spearman"], ascending=[True, False]).index[0]
naive_mae = summary.loc[reference, "mae"]
best_mae = summary.loc[best, "mae"]
random_mae = ru.random_ranking_mae(n_typical)
sig = significance.loc[best]

print(f"BEST MODEL: {best}")
print(f"  MAE {best_mae:.2f} positions | Spearman {summary.loc[best, 'spearman']:.2f} | "
      f"top-8 overlap {summary.loc[best, 'top_overlap']:.0%}")
print(f"  naive (previous season): MAE {naive_mae:.2f} | Spearman {summary.loc[reference, 'spearman']:.2f}")
print(f"  gain vs naive: {sig['mae_gain_vs_naive']:+.2f} positions, 95% CI [{sig['ci95_low']:+.2f}, {sig['ci95_high']:+.2f}]")
print(f"  skill vs random ranking: {1 - best_mae / random_mae:.0%} of the random-ranking error removed")
if sig["significant"]:
    print("  -> The model is reliably better than the naive baseline.")
else:
    print("  -> The advantage over the naive baseline is NOT statistically established on these seasons.")
print("\nIs it 'good'? A perfect ranking is impossible the day before the season (injuries, trades, luck, parity), "
      "so judge it against the naive baseline and the random reference above, not against 0.")

### Predicted vs actual ranking of the best model

In [ ]:
best_ranked = ranked[best]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(best_ranked["conf_rank"], best_ranked["pred_rank"], alpha=.4, s=20)
axes[0].plot([1, 15], [1, 15], "k--", lw=.8)
axes[0].set_xlabel("Actual rank"); axes[0].set_ylabel("Predicted rank"); axes[0].set_title(f"{best}: predicted vs actual"); axes[0].grid(alpha=.3)
bins = np.arange(-14.5, 15.5, 1)
axes[1].hist(best_ranked["err"], bins=bins, alpha=.6, label=best)
axes[1].hist(ranked[reference]["err"], bins=bins, alpha=.5, label=reference)
axes[1].set_xlabel("Predicted rank - actual rank"); axes[1].set_title("Error distribution"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

last_year = EVAL_YEARS[-1]
for conf, g in best_ranked[best_ranked["year"] == last_year].groupby("confID"):
    print(f"\nSeason {last_year} - conference {conf} - {best}")
    display(g.sort_values("pred_rank")[["pred_rank", "tmID", "conf_rank", "err", "score"]].rename(
        columns={"conf_rank": "actual_rank", "score": "predicted_points_z"}).round(2))

## 6. Predict one season

`predict_season(year)` trains the best model on every season before `year` and returns the two conference rankings
(the same thing the walk-forward did for each test season). To forecast a season not yet in the data, append the
new season's rows (team, conference, coach information) to `teams.csv` first - the features only look backwards.

In [ ]:
def predict_season(year, model_name=best):
    pred = ru.walk_forward(df, models[model_name], FEATURES, [year])
    out = ru.rank_predictions(pred, df)
    return out.sort_values(["confID", "pred_rank"])[["year", "confID", "pred_rank", "tmID", "score"]]

example = predict_season(EVAL_YEARS[-1])
display(example.head(15))

best_ranked.to_csv("ranking_predictions_output.csv", index=False)
print("Saved all test-season predictions of the best model -> ranking_predictions_output.csv")